<a href="https://colab.research.google.com/github/Htar-Su/_HTAR_Deep_RL_Course_by_Hugging_Face/blob/main/3_DeepRL_A2C_and_SAC_Robotic_Arm_Reach_and_PickandPlace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 6: Advantage Actor Critic (A2C) using Robotics Simulations with Panda-Gym 🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit8/thumbnail.png"  alt="Thumbnail"/>

In this notebook, you'll learn to use A2C with [Panda-Gym](https://github.com/qgallouedec/panda-gym). You're going **to train a robotic arm** (Franka Emika Panda robot) to perform a task:

- `Reach`: the robot must place its end-effector at a target position.

After that, you'll be able **to train in other robotics tasks**.


### 🎮 Environments:

- [Panda-Gym](https://github.com/qgallouedec/panda-gym)

###📚 RL-Library:

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to use **Panda-Gym**, the environment library.
- Be able to **train robots using A2C**.
- Understand why **we need to normalize the input**.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




## This notebook is from the Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 Study [Actor-Critic methods by reading Unit 6](https://huggingface.co/deep-rl-course/unit6/introduction) 🤗  

# Let's train our first robots 🤖

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process),  you need to push your trained model to the Hub and get the following results:

- `PandaReachDense-v3` get a result of >= -3.5.

To find your result, go to the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) and find your model, **the result = mean_reward - std of reward**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Create a virtual display 🔽

During the notebook, we'll need to generate a replay video. To do so, with colab, **we need to have a virtual screen to be able to render the environment** (and thus record the frames).

Hence the following cell will install the librairies and create and run a virtual screen 🖥

In [7]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [8]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

### Install dependencies 🔽

The first step is to install the dependencies, we’ll install multiple ones:
- `gymnasium`
- `panda-gym`: Contains the robotics arm environments.
- `stable-baselines3`: The SB3 deep reinforcement learning library.
- `huggingface_sb3`: Additional code for Stable-baselines3 to load and upload models from the Hugging Face 🤗 Hub.
- `huggingface_hub`: Library allowing anyone to work with the Hub repositories.

⏲ The installation can **take 10 minutes**.

In [9]:
# !pip install stable-baselines3[extra]
# !pip install gymnasium
# !pip install huggingface_sb3

# To fix the video rendering error, install compatible versions of Stable-Baselines3, Gymnasium, and huggingface_sb3.

!pip install gymnasium==0.29.1
!pip install panda_gym
!pip install stable-baselines3==2.3.2

!pip install huggingface_sb3==3.0
!pip install huggingface_hub


  Using cached panda_gym-3.0.7-py3-none-any.whl.metadata (4.3 kB)
  Using cached pybullet-3.2.7.tar.gz (80.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached panda_gym-3.0.7-py3-none-any.whl (23 kB)
  Created wheel for pybullet: filename=pybullet-3.2.7-cp313-cp313-linux_x86_64.whl size=99114372 sha256=697a09c042b5f5ec9e530485086999fb7ccbbaa9be91b1fa9b5bd5b13687aa91
  Stored in directory: /root/.cache/pip/wheels/8a/f0/6a/3f626b4ee04f8ec61e40021b095adf5359f9bd426065a368ac
Successfully built pybullet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [panda_gym]


In [10]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# save the package version
!pip freeze > /content/drive/MyDrive/models/requirements_a2c_PandaReachDense_v3.txt

# !pip freeze | grep -E "stable-baselines3|panda-gym|gymnasium|torch" > requirements.txt

In [ ]:
# To restore the package

!pip install -r /content/drive/MyDrive/models/requirements_a2c_PandaReachDense_v3.txt


## Import the packages 📦

In [11]:
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## PandaReachDense-v3 🦾

The agent we're going to train is a robotic arm that needs to do controls (moving the arm and using the end-effector).

In robotics, the *end-effector* is the device at the end of a robotic arm designed to interact with the environment.

In `PandaReach`, the robot must place its end-effector at a target position (green ball).

We're going to use the dense version of this environment. It means we'll get a *dense reward function* that **will provide a reward at each timestep** (the closer the agent is to completing the task, the higher the reward). Contrary to a *sparse reward function* where the environment **return a reward if and only if the task is completed**.

Also, we're going to use the *End-effector displacement control*, it means the **action corresponds to the displacement of the end-effector**. We don't control the individual motion of each joint (joint control).

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit8/robotics.jpg"  alt="Robotics"/>


This way **the training will be easier**.



### Create the environment

#### The environment 🎮

In `PandaReachDense-v3` the robotic arm must place its end-effector at a target position (green ball).

In [ ]:
env_id = "PandaReachDense-v3"

# Create the env
env = gym.make(env_id)

# Get the state space and action space
s_size = env.observation_space.shape
a_size = env.action_space

In [ ]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

The State Space is:  None
Sample observation OrderedDict({'achieved_goal': array([-2.981375 ,  9.658257 , -2.7046099], dtype=float32), 'desired_goal': array([8.696864, 5.334226, 9.826871], dtype=float32), 'observation': array([-9.901244 , -3.0178602, -5.1219163,  3.8371124, -0.9431574,
        9.536911 ], dtype=float32)})


The observation space **is a dictionary with 3 different elements**:
- `achieved_goal`: (x,y,z) the current position of the end-effector.
- `desired_goal`: (x,y,z) the target position for the end-effector.
- `observation`: position (x,y,z) and velocity of the end-effector (vx, vy, vz).

Given it's a dictionary as observation, **we will need to use a MultiInputPolicy policy instead of MlpPolicy**.

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

The Action Space is:  Box(-1.0, 1.0, (3,), float32)
Action Space Sample [ 0.08267706 -0.8052398  -0.78946626]


The action space is a vector with 3 values:
- Control x, y, z movement

### Normalize observation and rewards

A good practice in reinforcement learning is to [normalize input features](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html).

For that purpose, there is a wrapper that will compute a running average and standard deviation of input features.

We also normalize rewards with this same wrapper by adding `norm_reward = True`

[You should check the documentation to fill this cell](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecnormalize)

In [ ]:
env = make_vec_env(env_id, n_envs=4)

# Adding this wrapper to normalize the observation and the reward
env = # TODO: Add the wrapper


#### Solution

In [ ]:
env = make_vec_env(env_id, n_envs=4)

env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

### Create the A2C Model 🤖

For more information about A2C implementation with StableBaselines3 check: https://stable-baselines3.readthedocs.io/en/master/modules/a2c.html#notes

To find the best parameters I checked the [official trained agents by Stable-Baselines3 team](https://huggingface.co/sb3).

In [ ]:
model = # Create the A2C model and try to find the best parameters

#### Solution

In [ ]:
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1) # mean_reward on PandaReachDense-v3 --> -0.19 +/- 0.09


# model = A2C(
#     policy = "MultiInputPolicy",
#     env= env,
#     learning_rate=7e-4,
#     n_steps=8,
#     gamma=0.95,
#     gae_lambda=0.9,
#     ent_coef=0.0,
#     vf_coef=0.4,
#     max_grad_norm=0.5,
#     normalize_advantage=True,
#     use_sde=True,
#     sde_sample_freq=4,
#     policy_kwargs=dict(net_arch=[64, 64]),
#     tensorboard_log="./a2c_panda_tensorboard/",
#     verbose=1,
# ) # mean_reward on PandaReachDense-v3 --> -0.20 +/- 0.09

Using cuda device


### Train the A2C agent 🏃
- Let's train our agent for 1,000,000 timesteps, don't forget to use GPU on Colab. It will take approximately ~25-40min

In [ ]:
model.learn(1_000_000)

Streaming output truncated to the last 5000 lines.
|    std                | 0.383     |
|    value_loss         | 7.61e-05  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 2.73      |
|    ep_rew_mean        | -0.22     |
|    success_rate       | 1         |
| time/                 |           |
|    fps                | 374       |
|    iterations         | 23800     |
|    time_elapsed       | 1270      |
|    total_timesteps    | 476000    |
| train/                |           |
|    entropy_loss       | -1.22     |
|    explained_variance | 0.9788297 |
|    learning_rate      | 0.0007    |
|    n_updates          | 23799     |
|    policy_loss        | -0.000117 |
|    std                | 0.379     |
|    value_loss         | 0.000123  |
-------------------------------------
--------------------------------------
| rollout/              |            |
|    ep_len_mean        | 2.68     

In [ ]:
#Learning curve (open tensor board in a separate browser)

import subprocess
subprocess.Popen(["tensorboard", "--logdir", "./a2c_panda_tensorboard/", "--host", "0.0.0.0", "--port", "6006"])

# from google.colab import output
# output.serve_kernel_port_as_window(6006)

# Get just the URL instead of auto-opening
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(6006)")
print(url)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


https://6006-gpu-t4-s-kkb-usw4a1-2mkmpo3lj99kf-a.us-west4-1.prod.colab.dev


In [ ]:
# Save the model and  VecNormalize statistics when saving the agent
model.save("a2c-PandaReachDense-v3-1")
env.save("vec_normalize_a2c_PandaReachDense_v3-1.pkl")

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Save the model to Drive

model.save("/content/drive/MyDrive/models/3)_a2c-PandaReachDense-v3")
env.save("/content/drive/MyDrive/models/3)_vec_normalize_a2c_PandaReachDense_v3.pkl")

### Evaluate the agent 📈
- Now that's our  agent is trained, we need to **check its performance**.
- Stable-Baselines3 provides a method to do that: `evaluate_policy`

In [ ]:
import panda_gym
print(panda_gym.__version__)

3.0.7


In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load the saved statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize_a2c_PandaReachDense_v3-1.pkl", eval_env)
# eval_env = VecNormalize.load("/content/drive/MyDrive/models/3)_vec_normalize_a2c_PandaReachDense_v3.pkl", eval_env)

# We need to override the render_mode
eval_env.render_mode = "rgb_array"

#  do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
# model = A2C.load("a2c-PandaReachDense-v3-1")
# model = A2C.load("/content/drive/MyDrive/models/a2c-PandaReachDense-v3-1")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

Mean reward = -0.20 +/- 0.11


In [ ]:
print(eval_env.observation_space)
print(model.observation_space)

Dict('achieved_goal': Box(-10.0, 10.0, (3,), float32), 'desired_goal': Box(-10.0, 10.0, (3,), float32), 'observation': Box(-10.0, 10.0, (6,), float32))
Dict('achieved_goal': Box(-10.0, 10.0, (3,), float32), 'desired_goal': Box(-10.0, 10.0, (3,), float32), 'observation': Box(-10.0, 10.0, (6,), float32))


In [ ]:
# Create a separate environment for recording (by me)
from gymnasium.wrappers import RecordVideo
def make_env():
    return RecordVideo(
        gym.make(
            "PandaReachDense-v3",
            render_mode="rgb_array"
        ),
        video_folder="videos",
        episode_trigger=lambda episode: True,
    )

eval_env = DummyVecEnv([make_env])
eval_env = VecNormalize.load("vec_normalize_a2c_PandaReachDense_v3.pkl", eval_env)

eval_env.training = False
eval_env.norm_reward = False

# Load the model
model = A2C.load("a2c-PandaReachDense-v3")

# Run one episode manually
obs = eval_env.reset()

done = False
episode_reward = 0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)
    episode_reward += reward[0]
eval_env.close()
print("Episode reward:", episode_reward)


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Moviepy - Building video /content/videos/rl-video-episode-0.mp4.
Moviepy - Writing video /content/videos/rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/rl-video-episode-0.mp4


Moviepy - Building video /content/videos/rl-video-episode-1.mp4.
Moviepy - Writing video /content/videos/rl-video-episode-1.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/rl-video-episode-1.mp4
Episode reward: -0.24765089


In [1]:
from IPython.display import Video

Video("videos/rl-video-episode-0.mp4", embed=True)

### Publish your trained model on the Hub 🔥
Now that we saw we got good results after the training, we can publish our trained model on the Hub with one line of code.

📚 The libraries documentation 👉 https://github.com/huggingface/huggingface_sb3/tree/main#hugging-face--x-stable-baselines3-v20


By using `package_to_hub`, as we already mentionned in the former units, **you evaluate, record a replay, generate a model card of your agent and push it to the hub**.

This way:
- You can **showcase our work** 🔥
- You can **visualize your agent playing** 👀
- You can **share with the community an agent that others can use** 💾
- You can **access a leaderboard 🏆 to see how well your agent is performing compared to your classmates** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard


To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [19]:
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

3️⃣ We're now ready to push our trained agent to the 🤗 Hub 🔥 using `package_to_hub()` function

For this environment, **running this cell can take approximately 10min**

In [ ]:
# # To fix the video rendering error, reinstall compatible versions of Stable-Baselines3, Gymnasium, and huggingface_sb3.

# !pip install stable-baselines3==2.3.2
# !pip install gymnasium==0.29.1
# !pip install huggingface_sb3==3.0

In [ ]:
from huggingface_sb3 import package_to_hub

package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"Htar/a2c-{env_id}-no-hyperparameters", # Change the username
    commit_message="Initial commit",
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmpa1tqc5cv/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpa1tqc5cv/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpa1tqc5cv/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmpa1tqc5cv/-step-0-to-step-1000.mp4
ℹ Pushing repo Htar/a2c-PandaReachDense-v3-no-hyperparameters to the
Hugging Face Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:114: DeprecationWarning: hf_xet.upload_files() is deprecated. Use XetSession().new_upload_commit().start_upload_file() instead.
  return fn(*args, **kwargs)


  ...-v3/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

... (3 outputs left out)

  /tmp/tmpn0z9ytlg/replay.mp4 :  67%|######7   |  405kB /  604kB            

  ...0z9ytlg/vec_normalize.pkl:  67%|######7   | 1.77kB / 2.65kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/Htar/a2c-PandaReachDense-v3-no-hyperparameters/tree/main/


CommitInfo(commit_url='https://huggingface.co/Htar/a2c-PandaReachDense-v3-no-hyperparameters/commit/adf528f79a6dfd5ad5bc31ff9a7d40ed4f7d8b84', commit_message='Initial commit', commit_description='', oid='adf528f79a6dfd5ad5bc31ff9a7d40ed4f7d8b84', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Htar/a2c-PandaReachDense-v3-no-hyperparameters', endpoint='https://huggingface.co', repo_type='model', repo_id='Htar/a2c-PandaReachDense-v3-no-hyperparameters'), pr_revision=None, pr_num=None)

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! Why not trying  `PandaPickAndPlace-v3`?

If you want to try more advanced tasks for panda-gym, you need to check what was done using **TQC or SAC** (a more sample-efficient algorithm suited for robotics tasks). In real robotics, you'll use a more sample-efficient algorithm for a simple reason: contrary to a simulation **if you move your robotic arm too much, you have a risk of breaking it**.

PandaPickAndPlace-v1 (this model uses the v1 version of the environment): https://huggingface.co/sb3/tqc-PandaPickAndPlace-v1

And don't hesitate to check panda-gym documentation here: https://panda-gym.readthedocs.io/en/latest/usage/train_with_sb3.html

We provide you the steps to train another agent (optional):

1. Define the environment called "PandaPickAndPlace-v3"
2. Make a vectorized environment
3. Add a wrapper to normalize the observations and rewards. [Check the documentation](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecnormalize)
4. Create the A2C Model (don't forget verbose=1 to print the training logs).
5. Train it for 1M Timesteps
6. Save the model and  VecNormalize statistics when saving the agent
7. Evaluate your agent
8. Publish your trained model on the Hub 🔥 with `package_to_hub`


See you on Unit 7! 🔥

## Keep learning, stay awesome 🤗

===============================================================================

## **Advantage Actor Critic (A2C)** using Robotics Simulations with Panda-Gym **(PandaPickAndPlace-v3)**

SAC’s off-policy sample efficiency.
**Pick and Place** uses sparse rewards by default, which makes plain SAC struggle badly (it rarely stumbles onto success by chance). The standard fix — used in the original panda-gym benchmarks — is to pair SAC with **Hindsight Experience Replay (HER)**.

In [3]:
# 1 - 2
env_id = "PandaPickAndPlace-v3"
env = make_vec_env(env_id, n_envs=4)

# 3
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 4
# model = A2C(policy = "MultiInputPolicy",
#             env = env,
#             verbose=1) mean_reward -45.00 +/- 15.00

model = A2C(
    policy = "MultiInputPolicy",
    env= env,
    learning_rate=7e-4,
    n_steps=8,
    gamma=0.95,
    gae_lambda=0.9,
    ent_coef=0.0,
    vf_coef=0.4,
    max_grad_norm=0.5,
    normalize_advantage=True,
    use_sde=True,
    sde_sample_freq=4,
    policy_kwargs=dict(net_arch=[64, 64]),
    tensorboard_log="./a2c_panda_tensorboard/",
    verbose=1,
)

# 5
model.learn(1_000_000)

In [ ]:
# 6
model_name = "a2c-PandaPickAndPlace-v3_new";
model.save(model_name)
env.save("vec_normalize_a2c_PandaPickAndPlace_v3_new.pkl")

In [ ]:
# Save the model to Drive

model.save("/content/drive/MyDrive/models/4_a2c-PandaPickAndPlace-v3_with_Hyperparameters")
env.save("/content/drive/MyDrive/models/4_vec_normalize_a2c_PandaPickAndPlace_v3_with_Hyperparameters.pkl")

In [ ]:
# 6
# model_name = "a2c-PandaPickAndPlace-v3";
# model.save(model_name)
# env.save("vec_normalize_a2c_PandaPickAndPlace_v3.pkl")

# 7
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load the saved statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaPickAndPlace-v3")])
eval_env = VecNormalize.load("vec_normalize_a2c_PandaPickAndPlace_v3_new.pkl", eval_env)

#  do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
model = A2C.load(model_name)
mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# 8
package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"Htar/a2c-{env_id}-hyperparameters", # TODO: Change the username
    commit_message="Initial commit",
)

**==============================================================================**

## **Soft Actor Critic (SAC)** using Robotics Simulations with Panda-Gym (**PandaPickAndPlace-v3**)



## Import the packages 📦

In [12]:
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub

from stable_baselines3 import SAC
from stable_baselines3.her import HerReplayBuffer

from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

In [13]:
# 1 - 2
env_id = "PandaPickAndPlace-v3"
env = make_vec_env(env_id, n_envs=4)

# 3
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 4 Create the SAC Model
model = SAC(
    policy="MultiInputPolicy",
    env=env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    learning_rate=1e-3,
    buffer_size=1_000_000,
    batch_size=256,
    gamma=0.95,
    tau=0.05,
    train_freq=1,
    gradient_steps=1,
    learning_starts=1000,
    verbose=1,
)

# 5 Train the SAC agent
model.learn(total_timesteps=1_000_000)

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 441      |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
---------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 432      |
|    time_elapsed    | 0        |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 12       |
|    fps             | 429      |
|    time_elapsed    | 1        |
|    total_timesteps | 600      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 16       |
|    fps      

/usr/local/lib/python3.13/dist-packages/gymnasium/core.py:311: UserWarning: WARN: env.compute_reward to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.compute_reward` for environment variables or `env.get_wrapper_attr('compute_reward')` that will search the reminding wrappers.
  logger.warn(


Streaming output truncated to the last 5000 lines.
|    success_rate    | 0.29     |
| time/              |          |
|    episodes        | 23432    |
|    fps             | 139      |
|    time_elapsed    | 6865     |
|    total_timesteps | 959248   |
| train/             |          |
|    actor_loss      | 0.383    |
|    critic_loss     | 0.005    |
|    ent_coef        | 0.000227 |
|    ent_coef_loss   | 0.509    |
|    learning_rate   | 0.001    |
|    n_updates       | 239561   |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 38       |
|    ep_rew_mean     | -37.7    |
|    success_rate    | 0.31     |
| time/              |          |
|    episodes        | 23436    |
|    fps             | 139      |
|    time_elapsed    | 6866     |
|    total_timesteps | 959380   |
| train/             |          |
|    actor_loss      | 0.406    |
|    critic_loss     | 0.00558  |
|    ent_coef        | 0.00023 

In [14]:
# 6
model_name = "sac-PandaPickAndPlace-v3";
model.save(model_name)
env.save("vec_normalize_sac_PandaPickAndPlace_v3.pkl")

In [15]:
# Save the model to Drive

model.save("/content/drive/MyDrive/models/5_sac-PandaPickAndPlace-v3")
env.save("/content/drive/MyDrive/models/5_vec_normalize_sac_PandaPickAndPlace_v3.pkl")

In [20]:
# 7
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load the saved statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaPickAndPlace-v3")])
eval_env = VecNormalize.load("vec_normalize_sac_PandaPickAndPlace_v3.pkl", eval_env)

#  do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
model = SAC.load(model_name, env=eval_env)
mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# 8
package_to_hub(
    model=model,
    model_name=f"sac-{env_id}",
    model_architecture="SAC",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"Htar/sac-{env_id}-hyperparameters", # TODO: Change the username
    commit_message="Initial commit",
)

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean reward = -34.40 +/- 19.25
ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Saving video to /tmp/tmpjextsmng/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpjextsmng/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpjextsmng/-step-0-to-step-1000.mp4



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /tmp/tmpjextsmng/-step-0-to-step-1000.mp4
ℹ Pushing repo Htar/sac-PandaPickAndPlace-v3-hyperparameters to the
Hugging Face Hub


... (9 outputs left out)

  ...e-v3/critic.optimizer.pth:  10%|9         |  115kB / 1.19MB            

  /tmp/tmpscz776mv/replay.mp4 :  10%|9         | 64.1kB /  663kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/Htar/sac-PandaPickAndPlace-v3-hyperparameters/tree/main/


CommitInfo(commit_url='https://huggingface.co/Htar/sac-PandaPickAndPlace-v3-hyperparameters/commit/8ab8f652f81d9725132f75e0cd5f132354f6425c', commit_message='Initial commit', commit_description='', oid='8ab8f652f81d9725132f75e0cd5f132354f6425c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Htar/sac-PandaPickAndPlace-v3-hyperparameters', endpoint='https://huggingface.co', repo_type='model', repo_id='Htar/sac-PandaPickAndPlace-v3-hyperparameters'), pr_revision=None, pr_num=None)